# 0. Project Setup And Data Access Check

This notebook verifies the project folder structure and confirms that the raw UCI Diabetes dataset can be loaded successfully.

This notebook does not perform cleaning, modeling, feature engineering, leakage review, or intervention prioritization. Those steps are handled in later notebooks.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

OUTPUTS = PROJECT_ROOT / "outputs"
FIGURES = OUTPUTS / "figures"
MODEL_RESULTS = OUTPUTS / "model_results"
OUTREACH_LISTS = OUTPUTS / "outreach_lists"

for path in [
    DATA_RAW,
    DATA_PROCESSED,
    OUTPUTS,
    FIGURES,
    MODEL_RESULTS,
    OUTREACH_LISTS
]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root detected successfully.")
print("Project folder:", PROJECT_ROOT.name)

Project root detected successfully.
Project folder: 03_risk_stratification_intervention_prioritization


In [3]:
diabetes_path = DATA_RAW / "diabetic_data.csv"
mapping_path = DATA_RAW / "IDS_mapping.csv"

required_files = {
    "Diabetes data": diabetes_path,
    "ID mapping": mapping_path,
}

for name, path in required_files.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} not found at: {path}")

print("All required raw data files found.")

All required raw data files found.


In [4]:
df = pd.read_csv(diabetes_path)
ids_mapping = pd.read_csv(mapping_path)

print("Diabetes shape:", df.shape)
print("Mapping shape:", ids_mapping.shape)

Diabetes shape: (101766, 50)
Mapping shape: (67, 2)


In [5]:
# Verify that core project columns are available
required_columns = [
    "encounter_id",
    "patient_nbr",
    "race",
    "gender",
    "age",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "readmitted"
]

missing_required_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_required_columns:
    raise ValueError(f"Missing required columns: {missing_required_columns}")

print("All required project columns are present.")

All required project columns are present.


In [6]:
display(df.head())
display(ids_mapping.head())

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


,admission_type_id,description
0,1,Emergency
1,2,Urgent
2,3,Elective
3,4,Newborn
4,5,Not Available


In [7]:
print("Columns:")
print(df.columns.tolist())

print("\nTarget distribution:")
print(df["readmitted"].value_counts(dropna=False))

Columns:
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']

Target distribution:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64


In [8]:
# Preview the planned binary target for later notebooks
# This is a preview only; the official target engineering will happen in the cleaning notebook.

readmitted_30d_preview = (df["readmitted"] == "<30").astype(int)

print("Planned binary target: readmitted_30d")
print("Positive class definition: readmitted == '<30'")
print(f"30-day readmission count: {readmitted_30d_preview.sum():,}")
print(f"30-day readmission rate: {readmitted_30d_preview.mean():.3f}")

Planned binary target: readmitted_30d
Positive class definition: readmitted == '<30'
30-day readmission count: 11,357
30-day readmission rate: 0.112


In [9]:
print("Missing code count by column:")

missing_code_summary = (
    (df == "?")
    .sum()
    .to_frame("missing_code_count")
)

missing_code_summary["missing_code_pct"] = (
    missing_code_summary["missing_code_count"] / len(df)
)

missing_code_summary = (
    missing_code_summary
    .query("missing_code_count > 0")
    .sort_values("missing_code_pct", ascending=False)
)

display(missing_code_summary)

Missing code count by column:


,missing_code_count,missing_code_pct
weight,98569,0.968585
medical_specialty,49949,0.490822
payer_code,40256,0.395574
race,2273,0.022336
diag_3,1423,0.013983
diag_2,358,0.003518
diag_1,21,0.000206


## Missing Code Interpretation

The raw dataset uses `?` as a missing-value code in several fields.

This notebook only identifies missing-value patterns. Actual missing-value replacement, column-level cleaning decisions, and modeling decisions will be handled in Notebook 01.

High-missingness fields such as `weight`, `medical_specialty`, and `payer_code` should be reviewed carefully before modeling.